# 05.9 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/05-powerful-models/09-automl-pycaret/solutions.ipynb)

## Exercise 1

| PyCaret | scikit-learn |
|---|---|
| `setup(normalize=True)` | `Pipeline([SimpleImputer, StandardScaler, ...])` + `StratifiedKFold` |
| `compare_models()` | loop of `cross_validate` over models → DataFrame |
| `tune_model()` | `RandomizedSearchCV` (or Optuna) |
| `interpret_model()` | `shap.TreeExplainer(...)` + `shap.plots.beeswarm` |
| `finalize_model()` | `pipeline.fit(X_train, y_train)` on all training data |
| `save_model()` | `joblib.dump(pipeline, path)` |

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail
from sklearn.model_selection import train_test_split

raw = load_retail("Year 2010-2011")
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

cutoff = pd.Timestamp("2011-09-01")                       # features before, target after
past = sales[sales["InvoiceDate"] < cutoff]
future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < "2011-12-01")]

cust = past.groupby("Customer ID").agg(orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
                                       products=("StockCode", "nunique"),
                                       first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
cust["tenure_days"] = (cutoff - cust["first_order"]).dt.days
cust["days_since_last"] = (cutoff - cust["last_order"]).dt.days
future_spend = future.groupby("Customer ID")["Revenue"].sum().reindex(cust.index).fillna(0)
cust["churned"] = (future_spend == 0).astype(int)        # 1 = did not come back
cust = cust.drop(columns=["first_order", "last_order"])

features = ["orders", "spend", "products", "tenure_days", "days_since_last"]
X_train, X_test, y_train, y_test = train_test_split(
    cust[features], cust["churned"], test_size=0.25, random_state=42, stratify=cust["churned"])
print(f"{len(cust):,} customers, churn rate {cust['churned'].mean():.1%}, "
      f"train {len(X_train):,} / test {len(X_test):,}")

3,317 customers, churn rate 43.6%, train 2,487 / test 830


## Exercise 2

In [2]:
import lightgbm as lgb
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.model_selection import cross_val_score
zoo = [LogisticRegression(max_iter=1000), RandomForestClassifier(random_state=42, n_jobs=-1),
       ExtraTreesClassifier(random_state=42, n_jobs=-1), lgb.LGBMClassifier(verbose=-1, n_jobs=1)]
pd.Series({type(m).__name__: cross_val_score(make_pipeline(StandardScaler(), m), X_train, y_train,
                                             cv=5, scoring="roc_auc").mean() for m in zoo}
          ).sort_values(ascending=False).round(3)

LogisticRegression        0.743
LGBMClassifier            0.715
RandomForestClassifier    0.711
ExtraTreesClassifier      0.697
dtype: float64

## Exercise 3

In the PyCaret environment, the leaky column produces an AUC of essentially **1.000** for every model, and nothing warns you. The leaderboard looks wonderful. AutoML faithfully optimises whatever you give it, leaks included. Spotting that `leak` is just the answer in disguise is your job, and it is exactly what Modules 04 and 05.8 trained you to do (a feature with near-perfect importance is a red flag).